In [ ]:
%%capture
!pip install unsloth
# Unsloth'un güncel Qwen destekli versiyonu için
!pip install --force-reinstall --no-cache-dir --no-deps git+https://github.com/unslothai/unsloth.git

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!pip uninstall -y xformers
!pip install xformers

In [ ]:

import torch
from unsloth import FastLanguageModel
from datasets import load_dataset
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth.chat_templates import get_chat_template

INPUT_FILE = '/content/drive/MyDrive/restaurant_data/qwen_train.jsonl'

# 1. Modeli ve Tokenizer'ı 4-bit olarak yükle (RAM tasarrufu)
max_seq_length = 2048
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Qwen2.5-7B-Instruct",
    max_seq_length = max_seq_length,
    dtype = None,
    load_in_4bit = True,
)

# 2. QLoRA Adaptörlerini Ekle (Belirttiğin hiperparametreler)
model = FastLanguageModel.get_peft_model(
    model,
    r = 16, # Modelin öğrenme kapasitesi
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj"], # Hedef katmanlar
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 42,
    use_rslora = False,
    loftq_config = None,
)

# 3. Veri Setini Yükle ve ChatML Şablonunu Uygula
tokenizer = get_chat_template(
    tokenizer,
    chat_template = "chatml"
)

def formatting_prompts_func(examples):
    convos = examples["messages"]
    texts = [tokenizer.apply_chat_template(convo, tokenize=False, add_generation_prompt=False) for convo in convos]
    return { "text" : texts }

print("Eğitim verisi yükleniyor...")
dataset = load_dataset("json", data_files={"train": INPUT_FILE}, split="train")
dataset = dataset.map(formatting_prompts_func, batched = True)

# 4. Eğitimi Yapılandır ve Başlat
trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    dataset_num_proc = 2,
    packing = False,
    args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 10,
        num_train_epochs = 1.0,
        learning_rate = 2e-4,
        fp16 = not torch.cuda.is_bf16_supported(),
        bf16 = torch.cuda.is_bf16_supported(),
        logging_steps = 50,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 42,

        # --- DEĞİŞEN VE EKLENEN KISIMLAR ---
        output_dir = "/content/drive/MyDrive/restaurant_data/checkpoints", # Doğrudan Drive'a kaydet
        save_strategy = "steps",
        save_steps = 200, # Her 200 adımda bir Drive'a yedek (checkpoint) bırak
    ),
)

print("🚀 Eğitim Başlıyor...")
# Kaldığı yerden devam etmesi için bu parametreyi ekliyoruz:
trainer_stats = trainer.train(resume_from_checkpoint=True)
print(f"✅ Eğitim tamamlandı! Geçen süre: {trainer_stats.metrics['train_runtime']} saniye.")